# 05 · Train all three modelsOne notebook, three runs, identical data and hyperparameters everywhere exceptwhat is being tested.| Run | Starts from | Trains on | Purpose ||---|---|---|---|| **Stage 1** | `nllb600m-guz-init` (nb 04) | `stage1.jsonl` | learn Ekegusii — **the baseline** || **Stage 2** | **stage 1** | `stage2.jsonl` (PSA + 25% replay) | adapt to PSAs — **the result** || **Mixed** | `nllb600m-guz-init` | `mixed.jsonl` | control: was the ordering worth it? |Stage 2 starts from the stage-1 weights, which is the whole point of acurriculum. It also uses a **lower learning rate** — the model already knowsEkegusii and we are nudging its register, not teaching it a language fromscratch. A high LR here is the fastest way to destroy what stage 1 built.**Runtime** — roughly 3–5 h for stage 1, 1–2 h for stage 2, 4–6 h for the mixedcontrol. Each run checkpoints and can be resumed; you can also run stage 1 andstage 2, look at notebook 06, and come back for the control.**Inputs** — `artifacts/data/*.jsonl`, `artifacts/nllb600m-guz-init/`**Outputs** — three model directories under `artifacts/`

In [ ]:
import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd().parent))sys.path.insert(0, str(pathlib.Path.cwd()))import nb_common as CC.set_seed()C.use_house_style()print(f"project root: {C.ROOT}")

In [ ]:
import json, math, torch, numpy as npfrom torch.utils.data import Datasetfrom transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,                          Seq2SeqTrainer, Seq2SeqTrainingArguments,                          DataCollatorForSeq2Seq, EarlyStoppingCallback)C.gpu_report()C.require_files(C.DATA / "stage1.jsonl", C.DATA / "stage2.jsonl",                C.DATA / "mixed.jsonl", C.DATA / "dev.jsonl")assert C.EXTENDED_MODEL.exists(), "Run notebook 04 first"MIX = json.loads((C.DATA / "mixture.json").read_text())print("\nplanned runs:", {k: f"{v:,}" for k, v in MIX["counts"].items()})

## 1. HyperparametersTwo independent learning rates. `LR_STAGE1` teaches a new language and can bethe usual 5e-5. `LR_STAGE2` is deliberately 3× smaller: the model already speaksEkegusii and stage 2 only has ~23k examples, so a large LR would overwritestage 1 rather than refine it.Lower `BATCH` if you hit out-of-memory — the node is shared, so size against the*free* VRAM printed above.

In [ ]:
MAX_LEN      = 128BATCH        = 24ACCUM        = 2          # effective batch 48LR_STAGE1    = 5e-5LR_STAGE2    = 1.5e-5     # gentle: refine, do not overwriteEPOCHS_S1    = 3EPOCHS_S2    = 3WARMUP       = 0.03LABEL_SMOOTH = 0.1EVAL_EVERY   = 500RUN_STAGE1 = TrueRUN_STAGE2 = TrueRUN_MIXED  = True         # the control; set False to skip and add it laterprint(f"stage1 lr {LR_STAGE1} x{EPOCHS_S1} epochs | stage2 lr {LR_STAGE2} x{EPOCHS_S2}")print(f"effective batch {BATCH * ACCUM} | max_len {MAX_LEN}")

## 2. DatasetSpecial tokens are built by hand — `[lang_code] … [eos]` on both sides — as innotebook 04. The label sequence therefore begins with the target-language token,which is exactly what `forced_bos_token_id` produces at inference.Upsampling was already materialised in notebook 03, so this class is a plainencoder with no sampling logic hidden inside it.

In [ ]:
tok = AutoTokenizer.from_pretrained(C.EXTENDED_MODEL)EOS = tok.eos_token_iddef load_jsonl(path):    with open(path, encoding="utf-8") as fh:        return [json.loads(l) for l in fh]class TranslationDataset(Dataset):    def __init__(self, rows, max_len=MAX_LEN):        self.rows, self.max_len = rows, max_len    def _enc(self, text, lang):        ids = tok(text, add_special_tokens=False, truncation=True,                  max_length=self.max_len - 2)["input_ids"]        return [tok.convert_tokens_to_ids(lang)] + ids + [EOS]    def __len__(self): return len(self.rows)    def __getitem__(self, i):        r = self.rows[i]        return {"input_ids": self._enc(r["src"], r["src_lang"]),                "labels":    self._enc(r["tgt"], r["tgt_lang"])}dev_ds = TranslationDataset(load_jsonl(C.DATA / "dev.jsonl"))ex = TranslationDataset(load_jsonl(C.DATA / "stage1.jsonl"))[0]print(f"dev examples: {len(dev_ds):,}")print("source:", tok.decode(ex["input_ids"], skip_special_tokens=False)[:90])print("target:", tok.decode(ex["labels"], skip_special_tokens=False)[:90])

## 3. One training function, used three timesKeeping the runs in a single function is what makes the comparison meaningful:the only things that differ between them are the starting weights, the data andthe learning rate. Everything else is held constant by construction.

In [ ]:
def train_run(name, data_file, init_from, lr, epochs, out_dir):    print("=" * 70)    print(f"  {name}")    print(f"  init: {pathlib.Path(init_from).name}  data: {data_file}  lr: {lr}")    print("=" * 70)    rows = load_jsonl(C.DATA / data_file)    train_ds = TranslationDataset(rows)    model = AutoModelForSeq2SeqLM.from_pretrained(init_from)    model.config.max_length = MAX_LEN    collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100,                                      pad_to_multiple_of=8)    args = Seq2SeqTrainingArguments(        output_dir=str(C.ARTIFACTS / "checkpoints" / name),        per_device_train_batch_size=BATCH,        per_device_eval_batch_size=BATCH,        gradient_accumulation_steps=ACCUM,        learning_rate=lr,        num_train_epochs=epochs,        warmup_ratio=WARMUP,        label_smoothing_factor=LABEL_SMOOTH,        weight_decay=0.01,        lr_scheduler_type="linear",        bf16=torch.cuda.is_available(),        gradient_checkpointing=True,        group_by_length=True,        logging_steps=100,        eval_strategy="steps", eval_steps=EVAL_EVERY,        save_strategy="steps", save_steps=EVAL_EVERY,        save_total_limit=2,        load_best_model_at_end=True,        metric_for_best_model="eval_loss", greater_is_better=False,        predict_with_generate=False,        report_to="none", seed=C.SEED,    )    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,                             eval_dataset=dev_ds, data_collator=collator,                             callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])    print(f"  {len(train_ds):,} examples, "          f"~{math.ceil(len(train_ds)/(BATCH*ACCUM))*epochs:,} optimizer steps")    result = trainer.train()    trainer.save_model(str(out_dir))    tok.save_pretrained(out_dir)    print(f"  saved -> {out_dir}")    return trainer, result

## 4. Stage 1 — learn EkegusiiStarts from the tokenizer-extended checkpoint, where `guz_Latn` exists but hasonly Kikuyu's embedding. This run is where Ekegusii is actually learned, and itsoutput is the **baseline** every later number is compared against.

In [ ]:
history = {}if RUN_STAGE1:    t1, r1 = train_run("stage1", "stage1.jsonl", str(C.EXTENDED_MODEL),                       LR_STAGE1, EPOCHS_S1, C.STAGE1_MODEL)    history["stage1"] = t1.state.log_history    print(r1.metrics)else:    print("skipped (RUN_STAGE1 = False)")

## 5. Stage 2 — adapt to PSA registerStarts from **stage 1's weights**, not from the base model. Gentle learningrate, 25% Bible replay in the data. Watch the dev loss: if it rises sharply theLR is too high for this phase.

In [ ]:
if RUN_STAGE2:    assert C.STAGE1_MODEL.exists(), "Stage 1 must run first - stage 2 continues from it"    t2, r2 = train_run("stage2", "stage2.jsonl", str(C.STAGE1_MODEL),                       LR_STAGE2, EPOCHS_S2, C.STAGE2_MODEL)    history["stage2"] = t2.state.log_history    print(r2.metrics)else:    print("skipped (RUN_STAGE2 = False)")

## 6. Mixed control — everything at onceSame data as stage 1 and stage 2 combined, same starting point as stage 1, butin a single pass with no ordering. If the two-stage model beats this one, thecurriculum earned its place; if not, the simpler recipe is the honest answer.

In [ ]:
if RUN_MIXED:    t3, r3 = train_run("mixed", "mixed.jsonl", str(C.EXTENDED_MODEL),                       LR_STAGE1, EPOCHS_S1, C.MIXED_MODEL)    history["mixed"] = t3.state.log_history    print(r3.metrics)else:    print("skipped (RUN_MIXED = False) - notebook 06 will evaluate whatever exists")

## 7. Loss curves

In [ ]:
import matplotlib.pyplot as pltfig, axes = plt.subplots(1, len(history) or 1, figsize=(5 * max(1, len(history)), 4),                         squeeze=False)for ax, (name, hist) in zip(axes[0], history.items()):    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]    if tr: ax.plot(*zip(*tr), color=C.PALETTE[0], label="train")    if ev: ax.plot(*zip(*ev), color=C.PALETTE[1], marker="o", label="dev")    ax.set_title(name); ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax.legend()plt.tight_layout(); C.save_fig(fig, "05_loss_curves"); plt.show()C.save_json({name: [{k: v for k, v in h.items() if k in ("step", "loss", "eval_loss")}                    for h in hist] for name, hist in history.items()},            C.DATA / "training_history.json")print("Next: 06_evaluate.ipynb")